---
title: "Clustering"
number-sections: false
---

The goal of this notebook is not to master every detail of machine learning, but to understand the **whole game**. You’ll follow the complete CRISP-DM process, from Business Understanding to Evaluation, experiencing each phase in a simple and structured way. Deployment is the final phase of CRISP-DM; in this course it is limited to saving your trained model.

<figure style="text-align: center;">
    <img src="../../images/crispdm.png" alt="CRISP-DM" width="80%">
    <figcaption><em>Figure 1: The CRISP-DM process.</em></figcaption>
</figure>

In this notebook you will work through the main phases of a data science project:

* **Phase 1 – Business Understanding:** Explore why predicting molecular properties is valuable and what questions we want to answer.
* **Phase 2 – Data Understanding:** Learn how the molecular dataset is obtained and identify which columns are inputs (features) and which is the output (target).
* **Phase 3 – Data Preparation:** Prepare the dataset so it can be used for training models.
* **Phase 4 – Modelling:** Train a clustering model to form molecule groups
* **Phase 5 – Evaluation:** Evaluate the clusters to identify patterns

The goal here is not to perfect each step, but to connect them and see how the entire process fits together. By the end, you’ll have experienced the full machine learning workflow and gained insight into the lifecycle of a data science project.

Let’s get started!


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

print("pandas version:", pd.__version__)
print("numpy version:", np.__version__)
print("matplotlib version:", matplotlib.__version__)

## 1. Business Understanding

You've already covered this in the [Business Understanding](01_business_understanding.qmd) page.

## 2. Data Understanding

You've already covered this in the [Data Understanding](02_data_understanding.qmd) page. Simply load the dataset.

<mark>Your Turn</mark>

Read `starter_dataset.xlsx` as a pandas dataframe into the `df` variable

In [ ]:
# YOUR CODE HERE

## 3. Data Preparation

As explained in the Data Understanding section, the dataset is already prepared for you.

* We will not remove missing values in `homo`, `lumo`, or `stability`, since labels are not required for clustering.
* We will keep the `smiles` column. It will not be used as a model input, but it will be useful for evaluating the model and discovering patterns.

## 4. Modelling

In [ ]:
df.head()

For clustering, we only need the input features `X`, since this is an unsupervised learning task and there are no labels `y` to predict. In our dataset, the first column (`smiles`) is text that identifies the molecule, not a feature, and the last three columns contain labels (`homo`, `lumo`, `stability`) that we do not need for clustering. Therefore, we select only the descriptor columns in between to create `X`.

<mark>Your Turn</mark>

Create `X`.

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

To perform clustering, we will use the **HDBSCAN** algorithm. Start by importing it from `sklearn.cluster`, then create an `HDBSCAN` object with `copy=True` (this avoids a warning in recent scikit-learn versions), store it in a variable named `hdb`, and fit it to the feature matrix `X`. This will group similar data points into clusters without needing to specify the number of clusters in advance.

In [ ]:
# YOUR CODE HERE

## 5. Evaluation

In clustering, the evaluation phase is different from classification or regression because we do not have labels to check our results against. Instead, we evaluate how good the clusters are in two ways. First, we use measures such as the silhouette score or the elbow method to see if the clusters are compact and well separated. You learn both in the [Clustering](../../self-study/theme-3/04_clustering.ipynb) notebook of Self-study Theme 3 and use them in Iteration 3; in this notebook we skip this part. Second, we look at the clusters from a business or domain perspective to check if they make sense and are useful for the original question. In other words, evaluation is about both the **quality of the clusters** and their **usefulness in practice**.

In this project, our goal is to use clustering as a supporting step for the prediction tasks (classification and regression). For instance, clustering might reveal a molecular property that varies across clusters, which can then be added as a new feature to improve predictive models. Another application is clustering data from different sources to examine whether the source itself has an influence on the clustering patterns.

<mark>Your Turn</mark>

Retrieve the unique cluster labels assigned by the HDBSCAN model. Use `np.unique` on `hdb.labels_` and convert the result to a Python list to see all distinct cluster IDs.


In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

The label `-1` is not a cluster: HDBSCAN gives it to **noise**, molecules that are not close enough to others to belong to any cluster. Count how many molecules are in each cluster and how many are noise, for example with `pd.Series(hdb.labels_).value_counts()`.

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

HDBSCAN groups molecules that are close to each other, and it measures closeness as a distance computed over all 217 descriptors at once. Look at the ranges of the descriptors: compare the minimum and maximum of each column, for example with `X.describe().T` sorted by `max`. Which descriptor has by far the largest values? What does that mean for the distances between molecules, and so for your clusters?

Hint: cluster the molecules on that one descriptor alone and compare the labels with the ones you got from all descriptors.

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

Distance-based methods such as HDBSCAN need features on comparable scales. Scale `X` with `StandardScaler` (from `sklearn.preprocessing`), fit a new `HDBSCAN` model (again with `copy=True`) on the scaled data, and look at the clusters and the noise again. What changed? Use these clusters in the next step.

In [ ]:
# YOUR CODE HERE

<mark>Your Turn</mark>

Now evaluate the clusters using the second approach discussed above:

> Examine the clusters from a business or domain perspective to determine whether they make sense and provide value for the original question.

This means looking at each cluster and trying to understand why the points in that cluster belong together. Take a closer look at the smiles column in the dataset and compare it across the clusters.

- Do you notice any differences in the way the SMILES strings look?
- Can you think of a simple, quantitative property of these strings that might vary across clusters?
- Calculate this property for the whole dataset and for each cluster, and see if any patterns emerge.

If you can find such a property, you can use it as an input feature for your classification and regression models.

For example, you might calculate the percentage of molecules in each cluster that contain a Nitrogen atom and then check whether this proportion differs across clusters, revealing a feature you can feed into your classification and regression models.


In [ ]:
# YOUR CODE HERE